# stack-vs-cat composite — cx8: repeat a constant origin then stack with per-ray directions

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "stack-vs-cat"
DD_ATOM_IDS = ["einops-repeat", "stack-vs-cat"]
DD_SUBTOPICS = ["Einops: Repeat", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Many ARENA helpers expect rays in `(NR, 2, 3)` packed form — `[origin, direction]` stacked along a new middle axis. But the input data often comes split: a single shared origin `(3,)` plus a `(NR, 3)` bundle of directions.

The composition is:
  1. `einops-repeat` broadcasts the `(3,)` origin to `(NR, 3)` — stride-0 view.
  2. `stack-vs-cat` picks `torch.stack` (NOT cat!) because we need to INSERT a new axis of size 2, not extend an existing one.

Pick wrong (cat instead of stack) and you'd get `(NR, 6)` — same total elements, wrong shape. The atom recap is: stack inserts an axis, cat extends one.

### Composite Exercise — repeat a constant origin then stack with per-ray directions

**Atoms exercised together**: `einops-repeat`, `stack-vs-cat`

Implement `cx8_pack_rays(origin, directions)` — combine a single shared `origin: (3,)` with a per-ray `directions: (NR, 3)` into a packed rays tensor of shape `(NR, 2, 3)`.

1. **Repeat** the origin across the ray axis: `repeat(origin, 'd -> r d', r=NR)`. Result shape `(NR, 3)`.
2. **Stack vs cat**: the target shape is `(NR, 2, 3)` — you need a NEW axis of size 2 between `NR` and `3`. That's `torch.stack([..., ...], dim=1)`, not `torch.cat` (which would give `(NR, 6)`).

Return shape `(NR, 2, 3)`. The test asserts `rays[:, 0] == origin_broadcast` and `rays[:, 1] == directions`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx8_pack_rays(origin, directions):
    raise NotImplementedError

def _test_cx8():
    # atom-coverage: enforce that the solution actually uses einops.repeat
    # (not .expand() or t.broadcast_to or a full-copy tensor). Without this
    # check the value-tests pass with .expand and the einops-repeat atom
    # claim is fig-leaf.
    import inspect
    _src = inspect.getsource(cx8_pack_rays)
    assert 'repeat(' in _src, 'solution must use einops.repeat (not .expand/.broadcast_to/full-copy)'
    # Case A: cross-check against the manual reference.
    origin = t.tensor([1.0, 2.0, 3.0])
    directions = t.randn(5, 3)
    rays = cx8_pack_rays(origin, directions)
    assert tuple(rays.shape) == (5, 2, 3), f'shape: {tuple(rays.shape)}'
    # Slot 0 of every ray is the shared origin.
    for r in range(5):
        assert t.equal(rays[r, 0], origin), f'ray {r} origin mismatch: {rays[r, 0]}'
        assert t.equal(rays[r, 1], directions[r]), f'ray {r} direction mismatch: {rays[r, 1]}'

    # Case B: NEGATIVE check — cat would produce (NR, 6), not (NR, 2, 3).
    # Make sure the student did NOT just cat along dim=-1.
    assert rays.ndim == 3, f'expected 3-D (NR,2,3), got {rays.ndim}-D — did you use cat?'

    # Case C: realistic scale.
    origin2 = t.tensor([0.0, 0.0, 0.0])
    dirs2 = t.randn(200, 3)
    rays2 = cx8_pack_rays(origin2, dirs2)
    assert tuple(rays2.shape) == (200, 2, 3)
    assert t.allclose(rays2[:, 0], t.zeros(200, 3))
    assert t.equal(rays2[:, 1], dirs2)

    # Case D: hand-check on small tensor.
    o3 = t.tensor([7.0, 8.0, 9.0])
    d3 = t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
    out3 = cx8_pack_rays(o3, d3)
    expected = t.tensor([[[7.0, 8.0, 9.0], [1.0, 0.0, 0.0]],
                         [[7.0, 8.0, 9.0], [0.0, 1.0, 0.0]]])
    assert t.equal(out3, expected), f'got {out3}'
    _dd_passed.add('cx8')

_test_cx8()

<details><summary>Show solution — cx8</summary>

```python
def cx8_pack_rays(origin, directions):
    NR = directions.shape[0]
    # Atom A (einops-repeat): broadcast the shared origin to per-ray (NR, 3) — stride-0 view.
    origins = repeat(origin, 'd -> r d', r=NR)
    # Atom B (stack-vs-cat): we need a NEW axis of size 2 — that's stack (NOT cat).
    #   stack inserts an axis, cat extends one.
    return t.stack([origins, directions], dim=1)
```

The stack-vs-cat picker rule: count what you have vs what you want.
  - origins: (NR, 3); directions: (NR, 3). Two same-shape tensors.
  - target: (NR, 2, 3) — a NEW axis of size 2.
→ stack along dim=1.

If the target had been (NR, 6) — same axes but extended — that's cat. The mnemonic: len(inputs) becomes the NEW axis size for stack, but is summed into an EXISTING axis for cat.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["Einops: Repeat", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()